# LightGBM Classifier Prototype

Single-notebook workspace for the MALCA taxonomy LightGBM prototype. This replaces the temporary split ML modules while the model is being drafted.

Input table policy: the only accepted file suffix is `.parquet`.

Target policy: train one native LightGBM `Booster` per taxonomy target. That lets the notebook predict `morphology_primary` and `physical_family` side by side while keeping each model's class space explicit.


## 1. Setup

Edit the paths and training knobs here while prototyping.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import numpy as np
import pandas as pd

# Prototype knobs. Edit these in the notebook while iterating.
INPUT_PARQUET = Path("output/review/reviewed.parquet")
OUTPUT_DIR = Path("output/ml")
SEED = 42

# Taxonomy targets are trained as separate single-output LightGBM boosters.
TAXONOMY_TARGET_COLUMNS = [
    "morphology_primary",
    "physical_family",
    "morphology_secondary",
    "physical_subclass",
]
TARGET_COLS_TO_TRAIN = ["morphology_primary", "physical_family"]
PRIMARY_TARGET_COL = TARGET_COLS_TO_TRAIN[0]

ML_N_ESTIMATORS = 500
ML_LEARNING_RATE = 0.05
ML_NUM_LEAVES = 63
ML_SUBSAMPLE = 0.8
ML_COLSAMPLE_BYTREE = 0.8
ML_MIN_SAMPLES = 30
ML_CV_FOLDS = 5
ML_TOP_FEATURES = 20


## 2. Parquet Input

The ML prototype accepts explicit `.parquet` files only. For direct experimentation, you can also assign an in-memory DataFrame to `df`.


In [ ]:
def read_ml_parquet(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix != ".parquet":
        raise ValueError("ML input tables must be .parquet files")
    table = pd.read_parquet(path)
    if isinstance(table, pd.Series):
        return table.to_frame()
    return table


def load_training_table(path: str | Path = INPUT_PARQUET) -> pd.DataFrame:
    return read_ml_parquet(path)


In [ ]:
if INPUT_PARQUET.exists():
    df = load_training_table(INPUT_PARQUET)
    print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {INPUT_PARQUET}")
    display(df.head())
else:
    df = None
    print(f"Set INPUT_PARQUET to a .parquet review table. Current path does not exist: {INPUT_PARQUET}")


## 3. Feature Schema

Curated physics/context features plus deterministic categorical encoding.


In [ ]:
# ---- COLUMNS TO DROP -------------------------------------------------------
# These should **never** be used as ML features.  They are pipeline
# bookkeeping, redundant, or rule-based outputs that would bake in
# assumptions the model should learn on its own.
ML_DROP_COLUMNS: set[str] = {
    # Identifiers / paths (not physics)
    "candidate_id",
    "path",
    "lc_path",
    "source_path",
    "asas_sn_id",
    "gaia_id",
    "tmass_id",
    "allwise_id",
    "camera_ids",

    # Pipeline configuration (not physics)
    "trigger_mode",
    "trigger_type",
    "dip_trigger_threshold",
    "jump_trigger_threshold",
    "baseline_source",

    # Rule-based classification outputs
    # These encode hardcoded heuristics; the ML model should learn its
    # own decision boundaries rather than inheriting pipeline biases.
    "P_eb",
    "P_cv",
    "P_starspot",
    "P_disk",
    "final_class",
    "yso_class",

    # Filter flags — pipeline gating decisions, not physical features.
    # The ML model should learn its own gating from the raw data.
    "failed_any",
    "failed_posterior_strength",
    "failed_run_robustness",
    "failed_morphology",
    "failed_score",
    "failed_periodicity",
    "failed_gaia_ruwe",
    "failed_periodic_catalog",
    "failed_signal_amplitude",
    "bad_cameras_filtered",

    # Redundant pair — A_v = R_v * E(B-V); keep A_v_3d only
    "ebv_3d",

    # Review/taxonomy labels and metadata (not features)
    "interest_score",
    "review_pass",
    "notes",
    "status",
    "reviewer",
    "updated_at",
    "event_class",  # legacy derived taxonomy summary; not a feature
    "workflow_status",
    "disposition",
    "morphology_primary",
    "morphology_secondary",
    "morphology_polarity",
    "morphology_recurrence",
    "baseline_behavior",
    "physical_family",
    "physical_subclass",
    "classification_confidence",
    "priority_tags_json",
    "evidence_flags_json",
    "model_tags_json",
    "duplicate_of",
    "known_object_id",
    "known_object_source",
    "taxonomy_version",
    "legacy_review_json",

    # Timestamps (not useful as features directly)
    "jd_first",
    "jd_last",
    "dip_best_t0",
    "jump_best_t0",

    # Characterization module status columns
    "char_status_population",
    "char_error_population",
    "char_status_starhorse",
    "char_error_starhorse",
    "char_status_dust",
    "char_error_dust",
    "char_status_yso",
    "char_error_yso",
    "char_status_banyan",
    "char_error_banyan",
    "char_status_iphas",
    "char_error_iphas",
    "char_status_sfr",
    "char_error_sfr",
    "char_status_clusters",
    "char_error_clusters",
    "char_status_unwise",
    "char_error_unwise",

    # Payload JSON blob
    "payload_json",
    "imported_at",
    "source_id",
}

# ---- RECOMMENDED FEATURE COLUMNS -------------------------------------------
# These are the physics-driven features the classifier should use.
# Grouped by category for readability.  Order does not matter for the model.
ML_FEATURE_COLUMNS: list[str] = [
    # -- Periodicity --
    "periodic_flag",
    "periodicity_score",
    "lsp_power",
    "lsp_period",
    "lsp_bootstrap_sig",
    "lsp_is_alias",
    "lsp_is_significant",

    # -- Dip detection --
    "dip_significant",
    "dip_best_log_bf",
    "dip_best_delta_bic",
    "dip_best_width_param",
    "dip_symmetry_score",
    "dip_best_amp",
    "dip_best_alpha",
    "dip_best_tau",
    "dip_bayes_factor",
    "dip_best_p",
    "dip_best_mag_event",
    "dip_trigger_max",
    "dip_max_event_prob",

    # -- Dip runs --
    "dip_count",
    "dip_run_count",
    "dip_max_run_points",
    "dip_max_run_duration",
    "dip_max_run_sum",
    "dip_max_run_max",
    "dip_max_run_cameras",
    "dip_max_log_bf_local",

    # -- Dip recurrence --
    "dip_is_single_event",
    "dip_inter_event_spacing_median",
    "dip_inter_event_spacing_std",
    "dip_amplitude_consistency",
    "dip_duration_consistency",

    # -- Jump detection --
    "jump_significant",
    "jump_best_log_bf",
    "jump_best_delta_bic",
    "jump_best_width_param",
    "jump_best_amp",
    "jump_best_alpha",
    "jump_best_tau",
    "jump_bayes_factor",
    "jump_best_p",
    "jump_best_mag_event",
    "jump_trigger_max",
    "jump_max_event_prob",

    # -- Jump runs --
    "jump_count",
    "jump_run_count",
    "jump_max_run_points",
    "jump_max_run_duration",
    "jump_max_run_sum",
    "jump_max_run_max",
    "jump_max_run_cameras",
    "jump_max_log_bf_local",

    # -- Jump recurrence --
    "jump_is_single_event",
    "jump_inter_event_spacing_median",
    "jump_inter_event_spacing_std",
    "jump_amplitude_consistency",
    "jump_duration_consistency",

    # -- Event scoring --
    "dipper_score",
    "dipper_n_dips",
    "dipper_n_valid_dips",
    "jumper_score",
    "jumper_n_jumps",
    "jumper_n_valid_jumps",

    # -- Light curve basics --
    "n_points",
    "cadence_median_days",
    "n_cameras",
    "baseline_mag",

    # -- Stellar parameters (Gaia DR3) --
    "ruwe",
    "radial_velocity",
    "rv_amplitude_robust",
    "high_ruwe_flag",
    "teff_gspphot",
    "logg_gspphot",
    "mh_gspphot",
    "distance_gspphot",
    "parallax",
    "pmra",
    "pmdec",

    # -- Photometry --
    "tmass_j",
    "tmass_h",
    "tmass_k",
    "unwise_w1",
    "unwise_w2",
    "H_K",
    "W1_W2",
    "iphas_ha_mag",
    "unwise_w1_zscore",
    "unwise_w2_zscore",

    # -- Galactic coordinates (microlensing prior) --
    "gal_l",
    "gal_b",

    # -- Extinction & environment --
    "A_v_3d",
    "population",
    "age50",
    "mass50",
    "banyan_field_prob",
    "banyan_best_assoc",

    # -- Crossmatch context --
    "catalog_match",
    "vsx_class",
    "vsx_sep_arcsec",
    "sfr_name",
    "sfr_sep_arcmin",
    "cluster_name",
    "cluster_membership_prob",

    # -- Orbital / transit context --
    "a_circ_au",
    "transit_prob",
    "hill_radius_rsun",

    # -- Vetting (categorical — encoded as category codes) --
    "simbad_otype",
    "gaia_var_class",
    "asassn_var_type",
    "ztf_var_type",
    "alerce_lc_class",
    "tns_type",

    # -- Vetting (numeric) --
    "vetting_likely_known",
    "simbad_nbref",
    "simbad_sep_arcsec",
    "gaia_var_flag",
    "gaia_var_score",
    "gaia_eb_period",
    "gaia_eb_global_ranking",
    "gaia_epoch_n_obs",
    "gaia_epoch_g_range",
    "asassn_var_period",
    "ztf_var_period",
    "ztf_var_amp",
    "alerce_lc_prob",
    "alerce_ndet",
    "xray_det",
    "xray_flux",
    "pm_cluster_offset_sigma",

    # -- ALeRCE variability features --
    "stats_amplitude",
    "stats_beyond_1_std",
    "stats_con",
    "stats_delta_mag_fid",
    "stats_intrinsic_sigma_mag",
    "stats_first_mag",
    "stats_gskew",
    "stats_max_slope",
    "stats_meanvariance",
    "stats_median_abs_dev",
    "stats_median_brp",
    "stats_percent_amplitude",
    "stats_q31",
    "stats_skew",
    "stats_small_kurtosis",
    "stats_constancy_p_value",
    "stats_anderson_darling",
    "stats_pair_slope_trend",
    "stats_rcs",
    "stats_autocor_length",
    "stats_sf_ml_amplitude",
    "stats_sf_ml_gamma",

    # -- Harmonics (folded LC) --
    "stats_harmonics_order",
    "stats_harmonics_period",
    "stats_harmonics_a0",
    "stats_harmonics_model_amplitude",
    "stats_harmonics_reduced_chi2",
    "stats_harmonics_mag_1",
    "stats_harmonics_mag_2",
    "stats_harmonics_mag_3",
    "stats_harmonics_mag_4",
    "stats_harmonics_mag_5",
    "stats_harmonics_mag_6",
    "stats_harmonics_mag_7",
    "stats_harmonics_r21",
    "stats_harmonics_r31",
    "stats_harmonics_r41",
    "stats_harmonics_r51",
    "stats_harmonics_r61",
    "stats_harmonics_r71",
    "stats_harmonics_phase_2",
    "stats_harmonics_phase_3",
    "stats_harmonics_phase_4",
    "stats_harmonics_phase_5",
    "stats_harmonics_phase_6",
    "stats_harmonics_phase_7",
    "stats_harmonics_mse",
    "stats_psi_cs",
    "stats_psi_eta",

    # -- Stochastic models --
    "stats_gp_drw_sigma",
    "stats_gp_drw_tau",
    "stats_iar_phi",
    "stats_mhps_high",
    "stats_mhps_low",
    "stats_mhps_non_zero",
    "stats_mhps_pn_flag",
    "stats_mhps_ratio",
]

# ---- MORPHOLOGY FEATURE (categorical) --------------------------------------
# Best-fit morphology model name.  Encode as category codes for tree models.
ML_MORPH_COLUMNS: list[str] = [
    "dip_best_morph",
    "jump_best_morph",
]

# String/categorical feature columns requiring stable mapping across train/infer.
ML_CATEGORICAL_COLUMNS: list[str] = [
    "population",
    "banyan_best_assoc",
    "vsx_class",
    "sfr_name",
    "cluster_name",
    "simbad_otype",
    "gaia_var_class",
    "asassn_var_type",
    "ztf_var_type",
    "alerce_lc_class",
    "tns_type",
    "gaia_var_flag",
    *ML_MORPH_COLUMNS,
]


def infer_ml_feature_columns(
    df: pd.DataFrame,
    *,
    include_morph: bool = True,
) -> list[str]:
    """Return model feature columns present in *df* in stable order."""
    cols = [c for c in ML_FEATURE_COLUMNS if c in df.columns]
    if include_morph:
        cols += [c for c in ML_MORPH_COLUMNS if c in df.columns]
    return cols


def build_ml_feature_schema(
    df: pd.DataFrame,
    *,
    include_morph: bool = True,
) -> dict[str, Any]:
    """Fit a feature schema containing feature order and categorical maps."""
    features = infer_ml_feature_columns(df, include_morph=include_morph)
    categorical_columns = [c for c in features if c in ML_CATEGORICAL_COLUMNS]

    categorical_mappings: dict[str, dict[str, int]] = {}
    for col in categorical_columns:
        seen: set[str] = set()
        values: list[str] = []
        for value in df[col].tolist():
            if pd.isna(value):
                continue
            text = str(value)
            if text in seen:
                continue
            seen.add(text)
            values.append(text)
        values.sort()
        categorical_mappings[col] = {text: idx for idx, text in enumerate(values)}

    return {
        "features": features,
        "categorical_columns": categorical_columns,
        "categorical_mappings": categorical_mappings,
        "unknown_category_code": -1,
    }


def transform_ml_features(
    df: pd.DataFrame,
    feature_schema: dict[str, Any],
) -> pd.DataFrame:
    """Transform *df* into the model feature matrix using *feature_schema*."""
    features = [str(c) for c in feature_schema.get("features", [])]
    categorical_mappings = feature_schema.get("categorical_mappings", {}) or {}
    unknown_code = int(feature_schema.get("unknown_category_code", -1))

    if not features:
        return pd.DataFrame(index=df.index)

    X = pd.DataFrame(index=df.index)
    missing_series = pd.Series([pd.NA] * len(df), index=df.index, dtype="object")

    for col in features:
        if col in categorical_mappings:
            mapping = {str(k): int(v) for k, v in dict(categorical_mappings[col]).items()}
            raw = df[col] if col in df.columns else missing_series
            keys = raw.map(lambda v: None if pd.isna(v) else str(v))
            codes = keys.map(lambda v: mapping.get(v) if v is not None else np.nan).fillna(float(unknown_code))
            X[col] = pd.to_numeric(codes, errors="coerce")
        else:
            raw = df[col] if col in df.columns else np.nan
            X[col] = pd.to_numeric(raw, errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X


def select_ml_features(
    df: pd.DataFrame,
    *,
    include_morph: bool = True,
) -> pd.DataFrame:
    """Select and prepare ML-ready feature columns from *df*.

    - Keeps only columns listed in ``ML_FEATURE_COLUMNS`` (+ morph).
    - Encodes categorical columns with deterministic integer mappings.
    - Replaces inf with NaN, then fills NaN with 0.0.
    - Returns a copy; never mutates the input.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame (e.g. from ``export_reviews``).
    include_morph : bool
        If True (default), include ``dip_best_morph`` / ``jump_best_morph``
        as integer-coded categorical features.

    Returns
    -------
    pd.DataFrame
        Feature matrix ready for model training.
    """
    schema = build_ml_feature_schema(df, include_morph=include_morph)
    return transform_ml_features(df, schema)


In [ ]:
if df is not None:
    present_features = infer_ml_feature_columns(df)
    missing_features = [col for col in ML_FEATURE_COLUMNS if col not in df.columns]
    print(f"Usable configured features in this table: {len(present_features)}")
    print(f"Configured features missing from this table: {len(missing_features)}")
    display(pd.DataFrame({"present_feature": present_features}).head(30))
else:
    print("Load df first.")


## 4. Native LightGBM Helpers

LightGBM trains on numeric labels, so the string class mapping is tracked beside the native `Booster`.


In [ ]:
LIGHTGBM_MODEL_FILENAME = "lightgbm_booster.txt"
MODEL_METADATA_FILENAME = "model_metadata.json"


def predict_class_probabilities(booster: Any, x: pd.DataFrame, classes: list[str]) -> np.ndarray:
    probs = np.asarray(booster.predict(x), dtype=float)
    n_classes = len(classes)

    if probs.ndim == 1:
        if n_classes == 2 and probs.shape[0] == len(x):
            probs = np.column_stack([1.0 - probs, probs])
        elif n_classes and probs.size % n_classes == 0:
            probs = probs.reshape(-1, n_classes)
        else:
            raise ValueError("LightGBM probability output has an unexpected shape")

    if probs.ndim != 2 or probs.shape[1] != n_classes:
        raise ValueError("LightGBM class/probability shape mismatch")

    return probs


def predict_class_labels(booster: Any, x: pd.DataFrame, classes: list[str]) -> np.ndarray:
    probs = predict_class_probabilities(booster, x, classes)
    class_idx = np.argmax(probs, axis=1)
    return np.asarray(classes, dtype=object)[class_idx]


def feature_importances(booster: Any) -> np.ndarray:
    return np.asarray(booster.feature_importance(importance_type="split"))


## 5. Train and Evaluate

Each taxonomy target is a separate native LightGBM multiclass model. This is how we predict `morphology_primary` and `physical_family` simultaneously: train one booster per target, then append both sets of predictions to the scored table.

`classification_confidence` is reviewer confidence metadata, not the same thing as model probability. Use it later for filtering or weighting labels, not as the default target.


In [ ]:
def _build_lgb_params(seed: int, *, n_classes: int) -> dict[str, Any]:
    return {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "num_class": int(n_classes),
        "learning_rate": ML_LEARNING_RATE,
        "num_leaves": ML_NUM_LEAVES,
        "bagging_fraction": ML_SUBSAMPLE,
        "bagging_freq": 1 if ML_SUBSAMPLE < 1.0 else 0,
        "feature_fraction": ML_COLSAMPLE_BYTREE,
        "seed": seed,
        "bagging_seed": seed,
        "feature_fraction_seed": seed,
        "data_random_seed": seed,
        "verbosity": -1,
    }


def _encode_labels(y: pd.Series, classes: list[str]) -> np.ndarray:
    class_to_idx = {label: idx for idx, label in enumerate(classes)}
    encoded = y.map(class_to_idx)
    if encoded.isna().any():
        missing = sorted(set(y.loc[encoded.isna()].astype(str)))
        raise ValueError(f"Labels not present in class mapping: {missing}")
    return encoded.astype(np.int32).to_numpy()


def _balanced_sample_weights(y_encoded: np.ndarray, *, n_classes: int) -> np.ndarray:
    counts = np.bincount(y_encoded, minlength=n_classes)
    if np.any(counts == 0):
        raise ValueError("Cannot compute balanced weights when a class has no samples")
    return len(y_encoded) / (n_classes * counts[y_encoded])


def _train_native_model(
    x: pd.DataFrame,
    y: pd.Series,
    *,
    classes: list[str],
    seed: int,
) -> lgb.Booster:
    y_encoded = _encode_labels(y, classes)
    sample_weight = _balanced_sample_weights(y_encoded, n_classes=len(classes))
    train_set = lgb.Dataset(
        x,
        label=y_encoded,
        weight=sample_weight,
        feature_name=x.columns.tolist(),
        free_raw_data=True,
    )
    booster = lgb.train(
        _build_lgb_params(seed, n_classes=len(classes)),
        train_set,
        num_boost_round=ML_N_ESTIMATORS,
    )
    return booster


def _filter_training_rows(
    df: pd.DataFrame,
    *,
    label_col: str,
) -> pd.DataFrame:
    if label_col not in df.columns:
        raise ValueError(f"Missing label column: {label_col}")

    labels = df[label_col].astype("string").str.strip()
    valid = labels.notna() & (labels != "")
    out = df.loc[valid].copy()
    out[label_col] = labels.loc[valid].astype(str)
    return out


def _prepare_xy(
    df: pd.DataFrame,
    *,
    label_col: str,
    feature_schema: dict[str, Any] | None = None,
) -> tuple[pd.DataFrame, pd.Series, dict[str, Any]]:
    if label_col not in df.columns:
        raise ValueError(f"Missing label column: {label_col}")

    y = pd.Series(df[label_col], index=df.index).astype(str)
    schema = feature_schema or build_ml_feature_schema(df)
    x = transform_ml_features(df, schema)
    return x, y, schema


def _resolve_cv_folds(y: pd.Series, requested_folds: int) -> tuple[int, dict[str, int]]:
    class_counts = y.value_counts().sort_index()
    if len(class_counts) < 2:
        raise ValueError("Need at least 2 distinct classes to train a classifier.")

    min_class_count = int(class_counts.min())
    if min_class_count < 2:
        raise ValueError("Each class needs at least 2 labeled examples for cross-validation.")

    effective_folds = min(int(requested_folds), min_class_count)
    if effective_folds < 2:
        raise ValueError("Cross-validation requires at least 2 folds.")

    if effective_folds != int(requested_folds):
        print(
            f"Warning: reducing --cv-folds from {requested_folds} to {effective_folds} "
            f"because the smallest class has {min_class_count} samples"
        )

    return effective_folds, {str(k): int(v) for k, v in class_counts.items()}


def _run_cross_validation(
    x: pd.DataFrame,
    y: pd.Series,
    *,
    classes: list[str],
    seed: int,
    cv_folds: int,
) -> dict[str, Any]:
    splitter = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)
    oof_pred = pd.Series(index=y.index, dtype="object")
    fold_metrics: list[dict[str, Any]] = []

    for fold_idx, (train_idx, valid_idx) in enumerate(splitter.split(x, y), start=1):
        x_train, x_valid = x.iloc[train_idx], x.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = _train_native_model(x_train, y_train, classes=classes, seed=seed + fold_idx)
        pred_valid = pd.Series(predict_class_labels(model, x_valid, classes), index=y_valid.index)
        oof_pred.loc[y_valid.index] = pred_valid

        fold_metrics.append(
            {
                "fold": fold_idx,
                "n_train": int(len(train_idx)),
                "n_valid": int(len(valid_idx)),
                "accuracy": float(accuracy_score(y_valid, pred_valid)),
                "macro_f1": float(f1_score(y_valid, pred_valid, average="macro", zero_division=0)),
            }
        )

    oof_pred = oof_pred.fillna("<missing>")
    report_text = classification_report(y, oof_pred, labels=classes, zero_division=0)
    report_dict = classification_report(y, oof_pred, labels=classes, output_dict=True, zero_division=0)
    conf_mat = confusion_matrix(y, oof_pred, labels=classes)

    print("\nCross-validation classification report (OOF):")
    print(report_text)
    print("Confusion matrix (rows=true, cols=pred):")
    print(conf_mat)

    fold_acc = [m["accuracy"] for m in fold_metrics]
    fold_macro_f1 = [m["macro_f1"] for m in fold_metrics]

    return {
        "cv_folds": int(cv_folds),
        "cv_fold_metrics": fold_metrics,
        "cv_accuracy": float(np.mean(fold_acc)),
        "cv_accuracy_std": float(np.std(fold_acc)),
        "cv_macro_f1": float(np.mean(fold_macro_f1)),
        "cv_macro_f1_std": float(np.std(fold_macro_f1)),
        "cv_classification_report": report_dict,
        "cv_confusion_matrix": conf_mat.tolist(),
    }


def train_target_model(
    df: pd.DataFrame,
    *,
    label_col: str,
    seed: int = 42,
    cv_folds: int = ML_CV_FOLDS,
    min_samples: int = ML_MIN_SAMPLES,
) -> tuple[lgb.Booster, dict[str, Any], dict[str, Any]]:
    df_train = _filter_training_rows(df, label_col=label_col)
    if len(df_train) < int(min_samples):
        raise ValueError(
            f"Refusing to train: need at least {min_samples} labeled samples, got {len(df_train)}"
        )

    x, y, feature_schema = _prepare_xy(df_train, label_col=label_col)
    classes = sorted(y.unique().tolist())
    effective_folds, class_counts = _resolve_cv_folds(y, cv_folds)
    cv_metrics = _run_cross_validation(x, y, classes=classes, seed=seed, cv_folds=effective_folds)

    model = _train_native_model(x, y, classes=classes, seed=seed)
    train_pred = predict_class_labels(model, x, classes)
    train_acc = float(accuracy_score(y, train_pred))
    train_macro_f1 = float(f1_score(y, train_pred, average="macro", zero_division=0))

    top_features: list[dict[str, Any]] = []
    importances = list(zip(x.columns.tolist(), feature_importances(model).tolist()))
    importances.sort(key=lambda t: t[1], reverse=True)
    top_features = [
        {"feature": str(name), "importance": float(score)}
        for name, score in importances[:ML_TOP_FEATURES]
    ]

    metrics = {
        "target_col": str(label_col),
        "n_samples": int(len(df_train)),
        "n_features": int(x.shape[1]),
        "classes": classes,
        "class_counts": class_counts,
        "model_api": "lightgbm_native_dataset",
        "lightgbm_objective": "multiclass",
        "requested_cv_folds": int(cv_folds),
        **cv_metrics,
        "train_accuracy": train_acc,
        "train_macro_f1": train_macro_f1,
    }
    if top_features:
        metrics["feature_importance_top20"] = top_features

    return model, metrics, feature_schema


def save_model_artifacts(
    model: lgb.Booster,
    *,
    metrics: dict[str, Any],
    feature_schema: dict[str, Any],
    out_dir: Path,
) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    classes = [str(cls) for cls in metrics.get("classes", [])]
    if not classes:
        raise ValueError("Cannot save ML model artifacts without class metadata")

    model.save_model(str(out_dir / LIGHTGBM_MODEL_FILENAME))
    metadata = {
        "target_col": str(metrics.get("target_col", "target")),
        "classes": classes,
        "feature_names": [str(name) for name in model.feature_name()],
        "objective": str(metrics.get("lightgbm_objective", "multiclass")),
    }

    (out_dir / MODEL_METADATA_FILENAME).write_text(json.dumps(metadata, indent=2, sort_keys=True))
    (out_dir / "feature_schema.json").write_text(json.dumps(feature_schema, indent=2, sort_keys=True))
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))

def train_taxonomy_models(
    df: pd.DataFrame,
    *,
    target_cols: list[str] | tuple[str, ...],
    seed: int = SEED,
    cv_folds: int = ML_CV_FOLDS,
    min_samples: int = ML_MIN_SAMPLES,
) -> dict[str, dict[str, Any]]:
    trained: dict[str, dict[str, Any]] = {}
    for target_col in target_cols:
        model, metrics, feature_schema = train_target_model(
            df,
            label_col=target_col,
            seed=seed,
            cv_folds=cv_folds,
            min_samples=min_samples,
        )
        trained[str(target_col)] = {
            "model": model,
            "metrics": metrics,
            "feature_schema": feature_schema,
            "classes": metrics["classes"],
        }
    return trained


def save_taxonomy_model_artifacts(
    trained: dict[str, dict[str, Any]],
    *,
    out_dir: Path,
) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for target_col, artifact in trained.items():
        save_model_artifacts(
            artifact["model"],
            metrics=artifact["metrics"],
            feature_schema=artifact["feature_schema"],
            out_dir=out_dir / target_col,
        )


In [ ]:
if df is not None:
    trained_models = train_taxonomy_models(
        df,
        target_cols=TARGET_COLS_TO_TRAIN,
        seed=SEED,
        cv_folds=ML_CV_FOLDS,
        min_samples=ML_MIN_SAMPLES,
    )
    save_taxonomy_model_artifacts(trained_models, out_dir=OUTPUT_DIR)

    metrics_by_target = {target: artifact["metrics"] for target, artifact in trained_models.items()}
    summary_rows = []
    for target, metrics in metrics_by_target.items():
        summary_rows.append(
            {
                "target": target,
                "n_samples": metrics["n_samples"],
                "n_classes": len(metrics["classes"]),
                "cv_folds": metrics["cv_folds"],
                "cv_macro_f1": metrics["cv_macro_f1"],
                "train_macro_f1": metrics["train_macro_f1"],
            }
        )
    display(pd.DataFrame(summary_rows))

    if PRIMARY_TARGET_COL in trained_models:
        display(pd.DataFrame(trained_models[PRIMARY_TARGET_COL]["metrics"].get("feature_importance_top20", [])))
else:
    print("Load df first.")


## 6. Score Candidates

Load one saved booster per target and append target-specific columns such as `ml_predicted_morphology_primary`, `ml_prob_morphology_primary__dimming_event`, and `ml_predicted_physical_family`.


In [ ]:
def load_model(model_dir: Path) -> tuple[Any, list[str], dict[str, Any], dict[str, Any]]:
    import lightgbm as lgb

    model_dir = Path(model_dir)
    model_path = model_dir / LIGHTGBM_MODEL_FILENAME
    metadata_path = model_dir / MODEL_METADATA_FILENAME
    schema_path = model_dir / "feature_schema.json"

    model = lgb.Booster(model_file=str(model_path))
    metadata = json.loads(metadata_path.read_text())
    feature_schema = json.loads(schema_path.read_text())

    classes = metadata.get("classes")
    if not isinstance(classes, list) or not all(isinstance(cls, str) for cls in classes):
        raise ValueError(f"Invalid model metadata at {metadata_path}")

    required_keys = {"features", "categorical_mappings", "unknown_category_code"}
    if not isinstance(feature_schema, dict) or not required_keys.issubset(feature_schema):
        raise ValueError(f"Invalid feature schema at {schema_path}")

    return model, classes, feature_schema, metadata


def load_taxonomy_models(
    model_root: Path,
    *,
    target_cols: list[str] | tuple[str, ...] = TARGET_COLS_TO_TRAIN,
) -> dict[str, dict[str, Any]]:
    loaded: dict[str, dict[str, Any]] = {}
    for target_col in target_cols:
        model, classes, feature_schema, metadata = load_model(Path(model_root) / target_col)
        loaded[str(target_col)] = {
            "model": model,
            "classes": classes,
            "feature_schema": feature_schema,
            "metadata": metadata,
        }
    return loaded


def predict_target(
    model: Any,
    classes: list[str],
    feature_schema: dict[str, Any],
    df: pd.DataFrame,
    *,
    target_col: str,
) -> pd.DataFrame:
    x = transform_ml_features(df, feature_schema)
    if x.empty:
        raise ValueError("No model features found in input for prediction")

    probs = predict_class_probabilities(model, x, classes)
    preds = np.asarray(classes, dtype=object)[np.argmax(probs, axis=1)]

    result = pd.DataFrame(index=df.index)
    result[f"ml_predicted_{target_col}"] = preds
    for idx, cls in enumerate(classes):
        result[f"ml_prob_{target_col}__{cls}"] = probs[:, idx]
    return result


def predict_taxonomy_targets(
    models_by_target: dict[str, dict[str, Any]],
    df: pd.DataFrame,
) -> pd.DataFrame:
    result = df.copy()
    for target_col, artifact in models_by_target.items():
        target_predictions = predict_target(
            artifact["model"],
            artifact["classes"],
            artifact["feature_schema"],
            df,
            target_col=target_col,
        )
        result = pd.concat([result, target_predictions], axis=1)
    return result


In [ ]:
# Example scoring flow. Point CANDIDATE_PARQUET at another parquet table, or reuse INPUT_PARQUET.
CANDIDATE_PARQUET = INPUT_PARQUET

if CANDIDATE_PARQUET.exists() and OUTPUT_DIR.exists():
    candidate_df = read_ml_parquet(CANDIDATE_PARQUET)
    loaded_models = load_taxonomy_models(OUTPUT_DIR, target_cols=TARGET_COLS_TO_TRAIN)
    scored_df = predict_taxonomy_targets(loaded_models, candidate_df)
    display(scored_df.filter(regex="^(candidate_id|asas_sn_id|ml_predicted_|ml_prob_)").head())
else:
    print("Train/save models and set CANDIDATE_PARQUET before scoring.")
